# 05 — Sequence tagging: POS/NER with HMM, Viterbi, CRF template, BiLSTM tagger

This notebook is part of the NLP implementation pack for AI PMs, founders, and builders. It is designed to be runnable on toy data and easy to adapt to real company data.

## Mental model

Sequence tagging maps each input token to a label.

```text
tokens → labels per token
```

Examples:

- POS tagging
- NER
- clause extraction
- medical entity extraction
- product attribute extraction

This notebook implements HMM + Viterbi from scratch and provides CRF/BiLSTM templates.

In [ ]:
from pathlib import Path
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
print("Project root:", ROOT)

In [ ]:
import json
from collections import Counter, defaultdict
import numpy as np

examples = []
with open(DATA_DIR / "toy_ner.jsonl") as f:
    for line in f:
        examples.append(json.loads(line))
examples[0]

## HMM training

HMM components:

- hidden states = tags
- observations = words
- transition = P(tag_i | tag_{i-1})
- emission = P(word_i | tag_i)

In [ ]:
class HMMTagger:
    def __init__(self, alpha=0.1):
        self.alpha = alpha

    def fit(self, sequences):
        self.tag_counts = Counter()
        self.word_counts = Counter()
        self.transition_counts = Counter()
        self.emission_counts = Counter()
        self.tags = set()
        self.vocab = set()

        for ex in sequences:
            tokens, tags = ex["tokens"], ex["tags"]
            prev = "<START>"
            for word, tag in zip(tokens, tags):
                word = word.lower()
                self.tags.add(tag)
                self.vocab.add(word)
                self.tag_counts[tag] += 1
                self.word_counts[word] += 1
                self.transition_counts[(prev, tag)] += 1
                self.emission_counts[(tag, word)] += 1
                prev = tag
            self.transition_counts[(prev, "<END>")] += 1
        self.tags = sorted(self.tags)
        self.vocab.add("<UNK>")
        return self

    def transition_prob(self, prev, tag):
        possible = self.tags + (["<END>"] if tag == "<END>" else [])
        denom = sum(self.transition_counts[(prev, t)] for t in self.tags + ["<END>"]) + self.alpha * (len(self.tags) + 1)
        return (self.transition_counts[(prev, tag)] + self.alpha) / denom

    def emission_prob(self, tag, word):
        word = word.lower() if word.lower() in self.vocab else "<UNK>"
        denom = self.tag_counts[tag] + self.alpha * len(self.vocab)
        return (self.emission_counts[(tag, word)] + self.alpha) / denom

hmm = HMMTagger(alpha=0.1).fit(examples)
print("Tags:", hmm.tags)
print("P(B-ORG | <START>)", hmm.transition_prob("<START>", "B-ORG"))
print("P(commonwealth | B-ORG)", hmm.emission_prob("B-ORG", "commonwealth"))

## Viterbi decoding

Viterbi efficiently finds the best tag path without enumerating every possible sequence.

In [ ]:
def viterbi_decode(model, tokens):
    tags = model.tags
    n = len(tokens)
    dp = [{} for _ in range(n)]
    back = [{} for _ in range(n)]

    for tag in tags:
        dp[0][tag] = np.log(model.transition_prob("<START>", tag)) + np.log(model.emission_prob(tag, tokens[0]))
        back[0][tag] = None

    for i in range(1, n):
        for tag in tags:
            candidates = []
            for prev in tags:
                score = dp[i-1][prev] + np.log(model.transition_prob(prev, tag)) + np.log(model.emission_prob(tag, tokens[i]))
                candidates.append((score, prev))
            dp[i][tag], back[i][tag] = max(candidates)

    final_candidates = [(dp[n-1][tag] + np.log(model.transition_prob(tag, "<END>")), tag) for tag in tags]
    best_score, best_tag = max(final_candidates)

    path = [best_tag]
    for i in range(n - 1, 0, -1):
        path.append(back[i][path[-1]])
    path.reverse()
    return path, best_score

sentence = "OpenAI opened an office in Sydney".split()
tags, score = viterbi_decode(hmm, sentence)
list(zip(sentence, tags)), score

## Token-level feature extraction for CRF-style models

CRFs use rich features and model tag dependencies. In production you can use `sklearn-crfsuite`, `python-crfsuite`, or a neural CRF layer.

The feature function below is reusable for CRF, logistic token classification, or error analysis.

In [ ]:
def token_features(tokens, i):
    w = tokens[i]
    lower = w.lower()
    features = {
        "bias": 1.0,
        "word.lower": lower,
        "word[-3:]": lower[-3:],
        "word[-2:]": lower[-2:],
        "word.isupper": w.isupper(),
        "word.istitle": w.istitle(),
        "word.isdigit": w.isdigit(),
    }
    if i > 0:
        prev = tokens[i-1]
        features.update({
            "-1:word.lower": prev.lower(),
            "-1:word.istitle": prev.istitle(),
        })
    else:
        features["BOS"] = True
    if i < len(tokens)-1:
        nxt = tokens[i+1]
        features.update({
            "+1:word.lower": nxt.lower(),
            "+1:word.istitle": nxt.istitle(),
        })
    else:
        features["EOS"] = True
    return features

[token_features(examples[0]["tokens"], i) for i in range(2)]

In [ ]:
# Optional CRF implementation. Uncomment after installing sklearn-crfsuite.
# %pip install sklearn-crfsuite
# import sklearn_crfsuite
# from sklearn_crfsuite import metrics
#
# X = [[token_features(ex["tokens"], i) for i in range(len(ex["tokens"]))] for ex in examples]
# y = [ex["tags"] for ex in examples]
# crf = sklearn_crfsuite.CRF(algorithm="lbfgs", max_iterations=100, all_possible_transitions=True)
# crf.fit(X, y)
# y_pred = crf.predict(X)
# print(metrics.flat_classification_report(y, y_pred, zero_division=0))

## BiLSTM tagger template

BiLSTM produces contextual token representations. A CRF layer is often added on top to enforce globally consistent tag transitions.

This compact PyTorch model uses softmax per token for simplicity. Add a CRF layer for production-grade sequence consistency.

In [ ]:
try:
    import torch
    torch.set_num_threads(1)
    import torch.nn as nn
    from torch.nn.utils.rnn import pad_sequence

    tag_to_id = {tag: i for i, tag in enumerate(hmm.tags)}
    word_to_id = {"<PAD>": 0, "<UNK>": 1}
    for ex in examples:
        for tok in ex["tokens"]:
            word_to_id.setdefault(tok.lower(), len(word_to_id))

    def encode_example(ex):
        x = torch.tensor([word_to_id.get(tok.lower(), 1) for tok in ex["tokens"]], dtype=torch.long)
        y = torch.tensor([tag_to_id[tag] for tag in ex["tags"]], dtype=torch.long)
        return x, y

    encoded = [encode_example(ex) for ex in examples]

    class BiLSTMTagger(nn.Module):
        def __init__(self, vocab_size, tag_size, emb_dim=32, hidden_dim=32):
            super().__init__()
            self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
            self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
            self.classifier = nn.Linear(hidden_dim * 2, tag_size)

        def forward(self, x):
            out, _ = self.lstm(self.emb(x))
            return self.classifier(out)

    model = BiLSTMTagger(len(word_to_id), len(tag_to_id))
    print(model)
except Exception as e:
    print("PyTorch unavailable:", e)

In [ ]:
# Tiny training loop for demonstration. Real NER needs far more data.
try:
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(5):
        total = 0.0
        for x, y in encoded:
            logits = model(x.unsqueeze(0)).squeeze(0)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += float(loss.detach())
    test = "Google opened an office in Australia".split()
    x = torch.tensor([word_to_id.get(t.lower(), 1) for t in test]).unsqueeze(0)
    pred_ids = model(x).argmax(-1).squeeze(0).tolist()
    id_to_tag = {v:k for k,v in tag_to_id.items()}
    print(list(zip(test, [id_to_tag[i] for i in pred_ids])))
except Exception as e:
    print("Training skipped:", e)

## Product note

For a company entity extraction system:

1. Start with a labelled BIO dataset.
2. Build a regex/rule baseline for obvious entities.
3. Train a BERT token classifier or CRF baseline.
4. Measure entity-level precision/recall, not only token accuracy.
5. Human-review low-confidence or high-value extractions.